# Lab A — 用 `adk eval` 評保健品 agent（Colab 版）

延續你蓋的**保健品文案小組**（market研究 → 客群 → writer⇄reviewer → finalizer），
這一關**回頭評它**：走對流程沒（trajectory）＋ 產出夠好沒（response）。

> 為什麼是 notebook：`adk eval` 需要 agent 的程式碼在旁邊，所以第一步用 `git clone` 把整個 repo 抓下來，
> `lab2_multi_agent` 就在裡面，`adk eval` 對著它跑。全程 **Runtime → Run all** 就好。

## 0. 安裝（google-adk 的 `[eval]` 提供 adk eval）

In [ ]:
!pip install -q "google-adk[eval]==1.37.0"

## 1. 認證 + 填你自己的 GCP project

In [ ]:
from google.colab import auth
auth.authenticate_user()

PROJECT = "your-gcp-project"   # ← 改成你的 project id
print("project =", PROJECT)

## 2. 拿 agent 程式碼（clone repo）＋寫入 .env

clone 後 `lab2_multi_agent/`（agent）和 `lab_eval/`（考卷、門檻）就都在了。

In [ ]:
import os

!git clone -q https://github.com/amelielee-tech/adk-workshop.git

# adk 跑 agent 要呼叫 Vertex Gemini → 設定 project（adk eval 讀 .env、probe 讀 os.environ，兩個都給）
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "1"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT
os.environ["GOOGLE_CLOUD_LOCATION"] = "us-central1"
with open("adk-workshop/.env", "w") as f:
    f.write(f"GOOGLE_GENAI_USE_VERTEXAI=1\nGOOGLE_CLOUD_PROJECT={PROJECT}\nGOOGLE_CLOUD_LOCATION=us-central1\n")

print("repo cloned；.env 寫好，project =", PROJECT)
!ls adk-workshop/lab_eval/

## 3. 跑 `adk eval`（評 lab2_multi_agent）

一行指令拆解：`adk eval` **〈評誰〉〈用哪份考卷〉** `--config`**〈及格線〉**。
會真的把 agent 跑兩次（魚油、益生菌），約 1–2 分鐘。

In [ ]:
!cd adk-workshop && adk eval lab2_multi_agent lab_eval/copy_agent.evalset.json \
  --config_file_path lab_eval/test_config.json 2>&1 | grep -E "Tests passed|Tests failed|Using evaluation" 

## 4. 讀成乾淨的分數表（兩軸分開看）

上面那行只印通過數；這格把每個案例的兩個分數列清楚。

In [ ]:
import json, glob
import pandas as pd

rows = []
for f in sorted(glob.glob("adk-workshop/lab2_multi_agent/.adk/eval_history/*.json")):
    d = json.load(open(f))
    for c in d["eval_case_results"]:
        r = {"case": c["eval_id"]}
        for m in c["overall_eval_metric_results"]:
            r[m["metric_name"]] = round(m["score"], 3)
            r[m["metric_name"] + " → "] = "PASS" if m["eval_status"] == 1 else "FAIL"
        rows.append(r)

pd.DataFrame(rows)

**怎麼讀**：
- `tool_trajectory_avg_score`＝走對流程沒（有沒有先呼叫研究工具）。門檻 1.0、用 IN_ORDER。
- `response_match_score`＝文案字面（ROUGE-1 F1）像不像。**中文創作型天生低又飄**（所以門檻只 0.15）——這正是下一關 Lab B 要改用 LLM-judge 的理由。

## 5. 效率：打一次 agent 看 latency + token

adk eval 評品質，但 scorecard 還有一軸叫「效率」——只有真的跑一次才看得到。

In [ ]:
!cd adk-workshop && python lab_eval/probe_efficiency.py

## 收尾

- **agent 評估分兩軸**：走對流程（trajectory）＋ 產出夠好（response）——要一起讀。
- **每個指標都有極限**：ROUGE 對創作型中文很弱 → 所以 Lab B 用 LLM-judge。
- **效率**（latency/token）只有真的跑 agent 才量得到。

一句話：**eval 讓「感覺不錯」變成「量得出來」。** 下一關 Lab B：沒有標準答案時，怎麼評品質、又怎麼驗證你的評審。